# IsanAntiSpoof Kaggle + DagsHub Experiment
This notebook shows how to run the `IsanAntiSpoof` experiment in a Kaggle notebook and track metrics with DagsHub MLflow.

## 1. Import Required Libraries
Install repository dependencies and import helper modules.

In [ ]:
import os
from pathlib import Path
import subprocess
import sys

# Ensure git is available
!apt-get update && apt-get install -y git

# Install DagsHub early for tracking
!pip install dagshub


## 2. Load Dataset and Set Kaggle Paths
Define the repository and Kaggle dataset paths. If you are using a Kaggle dataset, mount it here.

In [ ]:
import os
from pathlib import Path
import shutil

os.chdir('/kaggle/working')

# Clone the GitHub repository if not already present
repo_url = 'https://github.com/ananyaknr/IsanAntiSpoof.git'
repo_dir = Path('/kaggle/working/IsanAntiSpoof')

if not repo_dir.exists():
    print(f'Cloning {repo_url}...')
    os.system(f'git clone {repo_url}')
else:
    print(f'Repository already exists at {repo_dir}')

root = repo_dir
print('Repo root:', root)
print('Exists:', root.exists())

kaggle_input = Path('/kaggle/input')
print('Kaggle input root exists:', kaggle_input.exists())
if kaggle_input.exists():
    print('Available Kaggle input datasets:')
    for p in sorted([p for p in kaggle_input.iterdir() if p.is_dir()]):
        print('-', p.name)

raw_root = root / 'data' / 'raw'
raw_root.mkdir(parents=True, exist_ok=True)
print('Local raw root:', raw_root)

mappings = {
    'asvspoof2019_la': ['asvspoof2019_la', 'asvspoof2019 la', 'asvspoof2019'],
    'Corpus-Spoof-genuine': ['corpus-spoof-genuine', 'corpus spoof genuine', 'corpus-spoof-genuine'],
    'Corpus-Spoof-VAJA': ['corpus-spoof-vaja', 'corpus spoof vaja', 'corpus-spoof-vaja'],
    'typhoon_isan': ['typhoon_isan', 'isan_bonafide', 'isan_bonafide_samples'],
    'isan_tts_spoofs': ['isan_tts_spoofs', 'isan_spoof', 'isan_tts']
}

for expected, patterns in mappings.items():
    target = raw_root / expected
    if target.exists():
        print(f'  [skip] {expected} already exists')
        continue

    source = None
    for candidate in sorted([p for p in kaggle_input.iterdir() if p.is_dir()]):
        lower_name = candidate.name.lower()
        if any(pattern.lower() in lower_name for pattern in patterns):
            source = candidate
            break

    if source is None:
        print(f'  [missing] no Kaggle input found for {expected}')
        continue

    print(f'  [map] {expected} -> {source}')
    try:
        target.symlink_to(source, target_is_directory=True)
        print(f'    symlink created for {expected}')
    except Exception as ex:
        print(f'    symlink failed for {expected}: {ex}')
        print(f'    copying files instead...')
        shutil.copytree(source, target, dirs_exist_ok=True)
        print(f'    copied {expected}')


In [ ]:
# Install project dependencies from the cloned repository
requirements_path = root / 'requirements.txt'
if requirements_path.exists():
    print(f'Installing requirements from {requirements_path}...')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(requirements_path)], check=False)
    print('Requirements installed successfully')
else:
    print(f'Requirements file not found at {requirements_path}')


## 3. Explore Dataset
Inspect the repository and dataset structure before training.

In [ ]:
if root.exists():
    for path in sorted(root.glob('**/*'))[:50]:
        print(path.relative_to(root))
else:
    print('Repo root not found. Clone the repo first.')

## 4. Preprocess Audio and Labels
Use the project scripts to build protocol metadata and extract features from raw audio.

In [ ]:
print('Protocol build / feature extraction will use raw data sourced from Kaggle input.')
print('Run the following if you need to generate the protocol or extract features:')
print('python src/data/build_protocol.py dataset.raw_root=data/raw')
print('python src/features/extract_all.py feature=lfcc')
print('python src/features/extract_all.py feature=mfcc')
print('python src/features/extract_all.py feature=cqcc')


## 5. Build the Anti-Spoofing Model
The repository uses `src/training/train.py` and Hydra to build GMM/LCNN/ResNet models.

## 5. Build the Anti-Spoofing Model
This repository uses `src/training/train.py` and Hydra to build GMM/LCNN/ResNet models.

### Experiment table
| ID | Repo config | Goal | Train + Validation | Test |
|---|---|---|---|---|
| E1 | `e1_baseline` | Standard Baseline | ASVspoof 2019 LA (≈4000 bona / 4000 spoof) | ASVspoof 2019 LA (≈1000 bona / 1000 spoof) |
| E1.5 | `e1_5_cross_lingual_gap` | Cross-Lingual Gap | ASVspoof 2019 LA | Thai dataset (AI bonafide + Thai spoof, ≈1000/1000) |
| E2 | `e2_dialect_gap` | Cross-Lingual + Dialect Gap | ASVspoof 2019 LA | Isan datasets (`typhoon_isan` + `isan_tts_spoofs`, ≈1000/1000) |
| E3 | `e3_cross_dialect_gap` | Cross-Dialect Gap | Thai dataset (`thaispoof` + Thai bonafide) | Isan datasets (`typhoon_isan` + `isan_tts_spoofs`, ≈1000/1000) |
| E4 | `e3_isan_aware` | Proposed Isan-Aware System | ASV + Thai + Isan training mix | Isan datasets (`typhoon_isan` + `isan_tts_spoofs`, ≈1000/1000) |
| E5 | `e4_feature_ablation` | Feature Ablation | Same as E4 | Same as E4 |
| E6 | `e5_model_ablation` | Model Ablation | Same as E4 | Same as E4 |


In [ ]:
# Duplicate experiment table placeholder cell - the actual table is in the previous markdown cell.


## 6. Train the Model
Run the experiment and send metrics to DagsHub MLflow. Replace the placeholders with your DagsHub org and token.

In [ ]:
os.environ['DAGSHUB_TOKEN'] = '<your-dagshub-token>'
dagshub_org = '<your-org>'
repo_name = 'IsanAntiSpoof'
mlflow_uri = f'https://dagshub.com/{dagshub_org}/{repo_name}.mlflow'
os.environ['MLFLOW_TRACKING_URI'] = mlflow_uri
print('MLflow URI:', mlflow_uri)

os.chdir(str(root))
print(f'Changed to {os.getcwd()}')

# Choose one experiment to run
experiment_name = 'e1_baseline'  # change this to e1_5_cross_lingual_gap, e2_dialect_gap, e3_cross_dialect_gap, e3_isan_aware, e4_feature_ablation, or e5_model_ablation

command = [
    'python', '-m', 'src.training.train',
    f'experiment={experiment_name}',
    f'mlflow.tracking_uri={mlflow_uri}',
    'mlflow.experiment_name=isan_antispoof',
    'dataset.raw_root=data/raw',
    'dataset.processed_root=data/processed',
    'hydra.run.dir=outputs/${now:%Y-%m-%d}/${now:%H-%M-%S}'
]
print('Running:', ' '.join(command))
subprocess.run(command, check=False)


## 6.1 Run All Experiments
Run the full experiment set sequentially using each experiment config file. This is useful for end-to-end benchmarking.


In [ ]:
experiments = [
    'e1_baseline',
    'e1_5_cross_lingual_gap',
    'e2_dialect_gap',
    'e3_cross_dialect_gap',
    'e3_isan_aware',
    'e4_feature_ablation',
    'e5_model_ablation'
]

for exp_name in experiments:
    print('\n=== Running', exp_name, '===')
    command = [
        'python', '-m', 'src.training.train',
        f'experiment={exp_name}',
        f'mlflow.tracking_uri={mlflow_uri}',
        'mlflow.experiment_name=isan_antispoof',
        'dataset.raw_root=data/raw',
        'dataset.processed_root=data/processed',
        'hydra.run.dir=outputs/${now:%Y-%m-%d}/${now:%H-%M-%S}'
    ]
    print('Running:', ' '.join(command))
    result = subprocess.run(command, check=False)
    print(f'{exp_name} finished with exit code', result.returncode)
    if result.returncode != 0:
        print('Stopping early due to failure.')
        break


## 7. Evaluate Model Performance
Read the aggregated results file produced by the experiment logger.

In [ ]:
results_path = root / 'experiments' / 'results.csv'
print('Results path:', results_path)
if results_path.exists():
    display(pd.read_csv(results_path).tail(10))
else:
    print('No results file found yet.')

## 8. Save Model and Prepare Submission
Locate trained checkpoints and save artifacts for download from Kaggle.

In [ ]:
checkpoints_dir = root / 'checkpoints'
print('Checkpoint directory exists:', checkpoints_dir.exists())
if checkpoints_dir.exists():
    for path in sorted(checkpoints_dir.glob('**/*'))[:50]:
        print(path.relative_to(root))
else:
    print('No checkpoints saved yet.')